In [1]:
import pandas as pd
import numpy as np
import json
import xgboost as xgb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import shap
import warnings
warnings.filterwarnings('ignore')

In [48]:
DATA_PATH   = 'Rental-Planner/outputs/features_trend.csv'
OUTPUT_DIR  = 'Rental-Planner/outputs'
TARGET         = 'target_yoy_pct'
CLIP_UPPER     = 25.4
CLIP_LOWER     = -15.0
RANDOM_STATE   = 42
CLIP_LOWER_OBSERVATIONS = 36

In [50]:
# Load the data and clip
df = pd.read_csv(DATA_PATH, low_memory=False, parse_dates=['date'])
df['zip_code'] = df['zip_code'].astype(str).str.zfill(5)
# Filter ZIPs with insufficient observations for reliable training
zip_counts = df.groupby('zip_code').size()
valid_zips = zip_counts[zip_counts >= 36].index
before_zips = df['zip_code'].nunique()
df = df[df['zip_code'].isin(valid_zips)]
print(f"  Dropped {before_zips - df['zip_code'].nunique()} ZIPs with <{CLIP_LOWER_OBSERVATIONS} observations")
print(f"  Rows after ZIP filter: {len(df):,}")
before_clip = df[TARGET].describe()
df[TARGET] = df[TARGET].clip(lower=CLIP_LOWER, upper=CLIP_UPPER)
print(f"  Rows affected: {((df[TARGET] == CLIP_UPPER) | (df[TARGET] == CLIP_LOWER)).sum():,}")

  Dropped 539 ZIPs with <36 observations
  Rows after ZIP filter: 181,312
  Rows affected: 2,141


In [60]:
zip_median = df.groupby('zip_code')['zori'].median()
luxury_threshold = zip_median.quantile(0.90)
luxury_zips = zip_median[zip_median >= luxury_threshold].index
df['is_luxury'] = df['zip_code'].isin(luxury_zips).astype(int)

In [61]:
# Select features

# Categorical features - will be target encoded
CAT_FEATURES = ['State', 'Metro']
 
# Numeric features
NUM_FEATURES = [
    # ZORI lags
    'zori_lag_1m', 'zori_lag_3m', 'zori_lag_6m',
    'zori_lag_12m', 'zori_lag_24m', 'zori_lag_36m',
    # Rolling stats
    'zori_roll_mean_6m', 'zori_roll_mean_12m', 'zori_roll_std_6m',
    # Rate of change
    'zori_mom_pct', 'zori_yoy_pct', 'zori_2y_pct',
    # SAFMR bedroom values
    'fmr_0br', 'fmr_1br', 'fmr_2br',
    # SAFMR lag and YoY
    'fmr_0br_lag1y', 'fmr_1br_lag1y', 'fmr_2br_lag1y',
    'fmr_0br_yoy_pct', 'fmr_1br_yoy_pct', 'fmr_2br_yoy_pct',
    # Calendar
    'month', 'quarter', 'month_sin', 'month_cos', 'year',
    # Luxury Flag
    'is_luxury'
]
 
# Keep only columns that exist
NUM_FEATURES = [f for f in NUM_FEATURES if f in df.columns]
CAT_FEATURES = [f for f in CAT_FEATURES if f in df.columns]

In [62]:
# Target Encoding for cat features
encoding_maps = {}
for col in CAT_FEATURES:
    means = df.groupby(col)[TARGET].mean()
    encoding_maps[col] = means.to_dict()
    df[f'{col}_encoded'] = df[col].map(means)
    NUM_FEATURES.append(f'{col}_encoded')

In [63]:
# Train / Test Split
X = df[NUM_FEATURES].copy()
y = df[TARGET].copy()
groups = df['zip_code']
 
splitter = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups))
 
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_test = groups.iloc[test_idx]

In [64]:
# Training
print("\nTraining XGBoost...")
model = xgb.XGBRegressor(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=10,   # to prevent overfitting on sparse zip codes
    reg_alpha=0.1,         # L1 regularization
    reg_lambda=1.0,        # L2 regularization
    random_state=RANDOM_STATE,
    n_jobs=-1,
    early_stopping_rounds=50,
    eval_metric='mae',
)
 
model.fit(
    X_train.to_numpy(), y_train.to_numpy(),
    eval_set=[(X_train.to_numpy(), y_train.to_numpy()), 
              (X_test.to_numpy(), y_test.to_numpy())],
    verbose=100,
)


Training XGBoost...
[0]	validation_0-mae:4.11103	validation_1-mae:4.14159
[100]	validation_0-mae:2.16200	validation_1-mae:2.27636
[200]	validation_0-mae:2.01604	validation_1-mae:2.19202
[300]	validation_0-mae:1.92229	validation_1-mae:2.14973
[400]	validation_0-mae:1.84666	validation_1-mae:2.12334
[500]	validation_0-mae:1.78341	validation_1-mae:2.10708
[600]	validation_0-mae:1.72311	validation_1-mae:2.09393
[700]	validation_0-mae:1.66937	validation_1-mae:2.08067
[799]	validation_0-mae:1.62170	validation_1-mae:2.07289


,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method

In [65]:
y_pred_train = model.predict(X_train.to_numpy())
y_pred_test  = model.predict(X_test.to_numpy())
 
def eval_metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"  {label}: MAE={mae:.3f}%  RMSE={rmse:.3f}%  R²={r2:.3f}")
    return {'mae': mae, 'rmse': rmse, 'r2': r2}
 
train_metrics = eval_metrics(y_train, y_pred_train, "Train")
test_metrics  = eval_metrics(y_test,  y_pred_test,  "Test ")
 
# Per-ZIP test evaluation
df_test = df.iloc[test_idx].copy()
df_test['predicted'] = y_pred_test
df_test['residual']  = df_test[TARGET] - df_test['predicted']
 
zip_eval = df_test.groupby('zip_code').apply(lambda g: pd.Series({
    'mae':   mean_absolute_error(g[TARGET], g['predicted']),
    'rmse':  np.sqrt(mean_squared_error(g[TARGET], g['predicted'])),
    'r2':    r2_score(g[TARGET], g['predicted']) if len(g) > 1 else np.nan,
    'n_obs': len(g),
    'state': g['State'].iloc[0],
    'metro': g['Metro'].iloc[0],
})).reset_index()
zip_eval = zip_eval[zip_eval['n_obs'] >= 10] 
print(f"\n  Median per-ZIP MAE: {zip_eval['mae'].median():.3f}%")
print(f"  Median per-ZIP R²:  {zip_eval['r2'].median():.3f}")
print(f"\n  Best 5 ZIPs by MAE:")
print(zip_eval.nsmallest(5, 'mae')[['zip_code','state','mae','r2','n_obs']].to_string(index=False))
print(f"\n  Worst 5 ZIPs by MAE:")
print(zip_eval.nlargest(5, 'mae')[['zip_code','state','mae','r2','n_obs']].to_string(index=False))

  Train: MAE=1.622%  RMSE=2.142%  R²=0.864
  Test : MAE=2.073%  RMSE=2.785%  R²=0.772

  Median per-ZIP MAE: 1.961%
  Median per-ZIP R²:  0.728

  Best 5 ZIPs by MAE:
zip_code state      mae        r2  n_obs
   73013    OK 0.861773  0.851611     91
   55369    MN 0.961916 -0.027292     42
   94066    CA 0.968547  0.684538     52
   21045    MD 1.055722  0.831689     91
   85234    AZ 1.062973  0.962178     91

  Worst 5 ZIPs by MAE:
zip_code state      mae        r2  n_obs
   79705    TX 6.705589 -0.494520     65
   77381    TX 5.177865  0.210601     88
   90014    CA 5.108086 -0.465262     49
   76102    TX 4.369867 -0.152599     54
   60005    IL 4.061350  0.363942     53


In [66]:
# SHAP Analysis
sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_test), min(2000, len(X_test)), replace=False)
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test.iloc[sample_idx].to_numpy())
 
shap_importance = pd.DataFrame({
    'feature':    NUM_FEATURES,
    'mean_shap':  np.abs(shap_values).mean(axis=0),
}).sort_values('mean_shap', ascending=False).reset_index(drop=True)
 
print("\n  Top 15 features by SHAP importance:")
print(shap_importance.head(15).to_string(index=False))


  Top 15 features by SHAP importance:
        feature  mean_shap
           year   2.932604
  Metro_encoded   0.942254
    zori_lag_1m   0.475667
    zori_2y_pct   0.429446
          month   0.411956
  State_encoded   0.397604
    zori_lag_3m   0.329403
   zori_yoy_pct   0.214704
   zori_lag_36m   0.214477
fmr_2br_yoy_pct   0.209576
        fmr_0br   0.189707
   zori_lag_24m   0.164360
   zori_mom_pct   0.154682
   zori_lag_12m   0.141540
fmr_0br_yoy_pct   0.133615


In [67]:
# Save Model
model.save_model(f'{OUTPUT_DIR}/model_xgb.json')
 
meta = {
    'features':        NUM_FEATURES,
    'cat_features':    CAT_FEATURES,
    'encoding_maps':   encoding_maps,
    'clip_upper':      CLIP_UPPER,
    'clip_lower':      CLIP_LOWER,
    'train_metrics':   train_metrics,
    'test_metrics':    test_metrics,
    'n_train_rows':    len(X_train),
    'n_test_rows':     len(X_test),
    'n_train_zips':    int(groups.iloc[train_idx].nunique()),
    'n_test_zips':     int(groups_test.nunique()),
}
with open(f'{OUTPUT_DIR}/model_features.json', 'w') as f:
    json.dump(meta, f, indent=2)
 
zip_eval.to_csv(f'{OUTPUT_DIR}/model_eval.csv', index=False)
shap_importance.to_csv(f'{OUTPUT_DIR}/shap_importance.csv', index=False)

In [69]:
# Second iteration evaluation
# CHANGES MADE:
# Removed bedroom ratio features (ratio_0br_to_2br through ratio_4br_to_2br)
## These belong in the inference layer for bedroom scaling, not in trend forecasting
## Were acting as noisy proxies for market type rather than contributing real signal

# Removed 3BR and 4BR SAFMR features (fmr_3br, fmr_4br, lags, and YoY)
## 3BR/4BR are largely owner-occupied in most markets -> introduces noise
## Kept 0BR, 1BR, 2BR as the most common rental unit types

# Kept 'year' as a feature after testing macro proxy alternatives
## Attempted replacement with national_median_zori and zori_vs_national
## R^2 dropped from 0.778 to 0.496 without year -> insufficient
## year is doing real work capturing macro regimes (COVID surge, normalization)
## Documented as known limitation: model relies on calendar year for macro context
## Consequence: model won't extrapolate reliably to years well beyond training range

# Added minimum observation filter (36 rows per ZIP in training data)
## Dropped 539 ZIPs with insufficient training history
## Removed zip codes like 46224 IN (29 obs, R^2=-7.0) that were actively distorting metrics

# Added is_luxury flag (top 10% by median ZORI)
## SHAP importance: 0.088 -> modest but shows contribution
## Ranks above quarter, rolling means, and cyclical calendar features
## Kept in final model

# Added minimum test observation filter (n_obs >= 10) for evaluation only
## Produces cleaner per zip code metrics without affecting model training

# IMPROVEMENTS OVER ITERATION 1:
# Test R^2: 0.778 -> 0.772 (marginal decrease, deemed acceptable given cleaner feature set)
# Median per-ZIP R^2: 0.611 -> 0.728 (improvement in per-market consistency)
# Worst ZIP MAE reduced from ~7% to ~6.7% range
# Feature set is cleaner and more defensible

# REMAINING CONCERNS:
# year still dominates SHAP at 2.93 vs 0.94 for Metro_encoded (next highest)
## Gap is wide but feature is legitimate -> document, don't remove
# Persistent worst performers are unique markets:
## 79705 TX (Midland)
## 90014 CA (downtown LA)
## 77381 TX (The Woodlands)
## These markets have drivers not capturable from ZORI and SAFMR alone
# Geographic clustering in worst performers shifted from Kansas City/Indianapolis to Texas

In [68]:
shap_df = pd.read_csv('Rental-Planner/outputs/shap_importance.csv')
print(shap_df.to_string())

               feature  mean_shap
0                 year   2.932604
1        Metro_encoded   0.942254
2          zori_lag_1m   0.475667
3          zori_2y_pct   0.429446
4                month   0.411956
5        State_encoded   0.397604
6          zori_lag_3m   0.329403
7         zori_yoy_pct   0.214704
8         zori_lag_36m   0.214477
9      fmr_2br_yoy_pct   0.209576
10             fmr_0br   0.189707
11        zori_lag_24m   0.164360
12        zori_mom_pct   0.154682
13        zori_lag_12m   0.141540
14     fmr_0br_yoy_pct   0.133615
15       fmr_1br_lag1y   0.131877
16             fmr_1br   0.128606
17       fmr_2br_lag1y   0.125845
18             fmr_2br   0.125555
19       fmr_0br_lag1y   0.121394
20         zori_lag_6m   0.115088
21     fmr_1br_yoy_pct   0.107923
22           is_luxury   0.088082
23    zori_roll_std_6m   0.061386
24  zori_roll_mean_12m   0.058230
25   zori_roll_mean_6m   0.049847
26             quarter   0.047659
27           month_sin   0.035202
28           m